### Day 1: Understand What Features You Need
  * Research waht drives wildfire spread - wind, slop, vegetation, humidity, temperature
  * Identify which data sources provide each features
  * Plan your feature engineering approach

#### How wildfires start
  * Three elements are needed for a fire to start:
    * fuel (wood. brush, lichen)
    * oxygen (from the air)
    * ignition source (heat from lightning or human activities)
  
#### How wildfires spread?
  * The primary factors that influence the spread of wildfires are
    * Fuels
    * Weather
    * Topography (what the landscape in the area is like)

* Fuel - Vegetation type and density - what's available to burn - MODIS Land Cover or ESA WorldCover
* Weather - Wind speed, wind direction, temperature, humidity - Open Meteo API
* Topography - Elevation and slope - fire moves faster uphill - USGS or NASA SRTM elevation data

In [1]:
# How wildfires start
  # Three elements are needed for a fire to start:
    # fuel (wood, brush, lichen)
    # oxygen (from the air)
    # ignition source (heat from lightning or human activities)

# How wildfires spread?
  # The primary factors that influence the spread of wildfires are
    # Fuels
    # Weather
    # Topography (what the landscape in the area is like)

### Day 2: Get Weather Data
  * Sign up for the Open-Meto API - free, no API key required
  * Pull historical weather data for your Canadian wildfire case study location
  * Key variables: wind speed, wind direction, temperature, humidity

In [11]:
import requests
import pandas as pd

In [19]:
# temperature_2m - air temperature at 2 meters above the ground in degrees Celsius
# Higher temperatures dry out vegetation and accelerate spread.

# relative_humidity_2m - relative humidity at 2 meters above ground as a percentage
# Lower humidity means drier fuel which burns faster
latitude = 59.9
longitude = -119.8
url = f"https://archive-api.open-meteo.com/v1/archive?latitude={latitude}&longitude={longitude}&start_date=2023-08-21&end_date=2023-09-30&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m"

In [20]:
response = requests.get(url, timeout=60).json()

In [21]:
hourly = response['hourly']

In [22]:
weather_df = pd.DataFrame(hourly)

In [23]:
weather_df['time'] = pd.to_datetime(weather_df['time'])

In [24]:
weather_df.head()

,time,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m
0,2023-08-21 00:00:00,23.3,28,14.1,220
1,2023-08-21 01:00:00,22.2,29,12.2,225
2,2023-08-21 02:00:00,20.7,39,6.9,227
3,2023-08-21 03:00:00,18.2,48,8.0,198
4,2023-08-21 04:00:00,16.6,54,11.4,204


In [25]:
weather_df.to_csv("../data/weather_data.csv", index=False)

### Day 3: Get Terrain Data
  * Download elevation data for your case study region
  * Calculate slope from elevation - fire moves faster uphill
  * Source: USGS National Elevation Dataset or NASA SRTM

In [ ]:
# Step 1 - Define Your Bounding Box
  # You need to tell the elevation library which geographic area to download data for. Your bounding box is the rectangular region around your 
  # Canadian wildfire case study:
  # South boundary: latitude 58
  # North boundary: latitude 62
  # West boundary: longitude -122
  # East boundary: longitude -118

In [68]:
south_boundary, north_boundary, west_boundary, east_boundary = 59, 61, -121, -119

In [ ]:
# Step 2 - Load your API key securely
  # using dotenv to read it from your .env file instead of hardcoding it

In [67]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OEPNTOPOGRAPHY_API_KEY')

In [57]:
# Step 3 - Call the OpenTopography API
  # pass your bounding box and API key as parameters using requests, same pattern as your Open-Meteo call

In [65]:
import requests

In [69]:
query_params = {
  "demtype": "SRTMGL1",
  "south": south_boundary,
  "north": north_boundary,
  "west": west_boundary,
  "east": east_boundary,
  "outputFormat": "GTiff",
  "API_Key": api_key
}

In [70]:
response = requests.get(url="https://portal.opentopography.org/API/globaldem", params=query_params)

In [ ]:
# Step 4 - Save the response as a GeoTIFF file
  # the API returns raw binary data that you write directly to ../data/elevation.tif

In [77]:
with open('../data/elevation.tif', "wb") as elevation_data_file:
    elevation_data_file.write(response.content)

In [60]:
# Step 5 - Load the GeoTIFF using rasterio
  # read the elevation values into a NumPy array

In [89]:
import rasterio # an open-source Python library used to read, write, and analyze geospatial raster data

In [145]:
with rasterio.open('../data/elevation.tif') as elevation:
    print(elevation.nodata)
    
    # Read the elevation values from the first band into a NumPy array
    band1 = elevation.read(1)
    
    # Print the shape of the array to confirm it loaded correctly
    print(band1.shape)

-32768.0
(7200, 7200)


In [167]:
import numpy as np

In [187]:
# Convert band1 to float64 first
band1_float = np.array(band1, dtype=float)

In [188]:
(band1_float == elevation.nodata).sum()

np.int64(25920000)

In [189]:
band1_float[band1_float == elevation.nodata] = np.nan

In [190]:
np.isnan(band1_float).sum()

np.int64(25920000)

In [191]:
band1_float

array([[ nan,  nan,  nan, ...,  nan,  nan,  nan],
       [ nan,  nan,  nan, ...,  nan,  nan,  nan],
       [ nan,  nan,  nan, ...,  nan,  nan,  nan],
       ...,
       [641., 641., 641., ..., 373., 372., 372.],
       [641., 641., 642., ..., 372., 371., 371.],
       [641., 641., 642., ..., 371., 371., 371.]], shape=(7200, 7200))

In [ ]:
gradient_north_south, gradient_east_west = np.gradient(band1_float) # computing the gradients of how the steep the terrain is across the north south and east west range respectively

In [193]:
gradient_north_south_squared = gradient_north_south ** 2

In [194]:
gradient_east_west_squared = gradient_east_west ** 2

In [195]:
slope = np.sqrt(gradient_north_south_squared + gradient_east_west_squared)

In [196]:
np.nanmin(slope), np.nanmax(slope)

(np.float64(0.0), np.float64(13.583077707206124))

In [ ]:
# Step 7 - Visualize
  # Plot both the elevation and slope as heatmaps uisng Matploblib so you can see the terrain of your case study region visually.
  # Mountainous areas will show up clearly

In [62]:
# Step 8 - Save
  # Save both the elevation and slop arrays so you can merge them with your fire detection
  # data later in the week.

### Day 4: Get Vegetation Data
  * Download land cover / vegetation type data
  * Source: MODIS Land Cover product (MCD12Q1) or ESA WorldCover
  * Different vegetation types burn at different rates and intensities

### Day 5: Join All Data Sources
  * Merge weather, terrain, and vegetation data onto your fire detection dataframe
  * Each fire detection row should now have wind, slope, and vegetation attributes

### Day 6: Feature Engineering
  * Create derived features - wind alignment with slope, days since last rain, vegetation dryness index
  * These compound features often matter more than raw variables

### Day 7 - Wrap Up Week 2
  * Commit everything to GitHub
  * Document your feature set clearly in a notebook markdown cell